# 08 — Full-Context Linear-Attention DT

Same model as `05_train_linearattn_dt.ipynb` (`LinearDecisionTransformer`, `phi(x) = elu(x) + 1` causal linear attention), but each training example is **one full session** instead of a sliding 32-step window.

Motivation: the suffix-sum RTG signal references every reward to the end of the session — across trial boundaries — but a 32-step window almost never contains the trial boundary or the rewards the RTG is pointing at. Linear attention's `O(L)` cost makes the full session affordable; this notebook tests whether a single-stream model with full context already closes the cross-trial gap (the cheaper half of the ablation pair, paired with notebook 09's hybrid).

Right-pad each session to `FULL_CTX`; mask loss at padded positions. Sessions longer than `FULL_CTX` are truncated to their tail (the terminal reward step is the most informative for RTG semantics).


## 1. Setup


In [1]:
from pathlib import Path
import json, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import sys; sys.path.insert(0, '../scripts')
import train_dt as T  # reuses encode_session for per-step encoding

from corner_maze_rl.encoders.grid_cells import GridCellEncoder
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.models.linear_decision_transformer import (
    LinearDTConfig, LinearDecisionTransformer,
)

DATA_PATH    = Path('../data/yoked/dataset/actions_synthetic_pretrial.parquet')
RUN_DIR      = Path('../runs/dt/nb08'); RUN_DIR.mkdir(parents=True, exist_ok=True)

FULL_CTX     = 4096     # covers ~99% of sessions; longer sessions truncated to tail
EMBED_DIM    = 60       # matched to GridCellEncoder.output_dim
NUM_HEADS    = 4        # d_head = 60/4 = 15
NUM_LAYERS   = 2
POS_ENCODING = 'learned'
LR           = 5e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 4        # full sessions are big; bumping risks OOM at L=12288 tokens.
EPOCHS       = 5        # full-context is expensive — fewer epochs.
VAL_FRAC     = 0.10
SEED         = 0
MAX_SESSIONS = None     # None = all; set e.g. 50 for a smoke run

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device={device}  d_head={EMBED_DIM // NUM_HEADS}  FULL_CTX={FULL_CTX}')


device=cuda  d_head=15  FULL_CTX=4096


## 2. Load and pack each session as one example


In [2]:
df = pd.read_parquet(DATA_PATH)
if MAX_SESSIONS is not None:
    keep = sorted(df['session_id'].unique())[:MAX_SESSIONS]
    df = df[df['session_id'].isin(keep)].reset_index(drop=True)

encoder = GridCellEncoder()
assert encoder.output_dim == EMBED_DIM

# 11*11*4 = 484 distinct (x, y, direction) poses — store pose IDs (int16) and
# look up the 60-D vector on-GPU at batch time. With sessions right-padded to
# FULL_CTX=4096 that's 549*4096*int16 ≈ 4 MB vs the materialized state-vector
# tensor's 549*4096*60*float32 ≈ 540 MB.
N_X, N_Y, N_D = 11, 11, 4
N_POSES = N_X * N_Y * N_D
pose_table_np = np.zeros((N_POSES, EMBED_DIM), dtype=np.float32)
for x in range(1, N_X + 1):
    for y in range(1, N_Y + 1):
        for d in range(N_D):
            pose_table_np[((x - 1) * N_Y + (y - 1)) * N_D + d] = encoder.encode(x, y, d)
pose_table = torch.from_numpy(pose_table_np).to(device)

def pack_session_ids(sdf, full_ctx):
    if len(sdf) > full_ctx:
        sdf = sdf.iloc[-full_ctx:]
    n = len(sdf)
    xs = sdf['grid_x'].to_numpy(np.int32)
    ys = sdf['grid_y'].to_numpy(np.int32)
    ds = sdf['direction'].to_numpy(np.int32)
    pose_orig = (((xs - 1) * N_Y + (ys - 1)) * N_D + ds).astype(np.int16)
    rewarded = sdf['rewarded'].to_numpy(np.float32)
    rtg_orig = np.flip(np.cumsum(np.flip(rewarded))).astype(np.float32)
    action_orig = sdf['action'].to_numpy(np.int64).clip(0, 4).astype(np.int8)
    pose_ids = np.zeros(full_ctx, dtype=np.int16);   pose_ids[:n]   = pose_orig
    action_ids = np.zeros(full_ctx, dtype=np.int8);  action_ids[:n] = action_orig
    rtg = np.zeros(full_ctx, dtype=np.float32);      rtg[:n]        = rtg_orig
    mask = np.zeros(full_ctx, dtype=np.float32);     mask[:n]       = 1.0
    return pose_ids, action_ids, rtg, mask

sids = df['session_id'].unique().tolist()
poses_all = np.empty((len(sids), FULL_CTX), dtype=np.int16)
acts_all  = np.empty((len(sids), FULL_CTX), dtype=np.int8)
rtgs_all  = np.empty((len(sids), FULL_CTX), dtype=np.float32)
masks_all = np.empty((len(sids), FULL_CTX), dtype=np.float32)
for i, sid in enumerate(sids):
    sdf = df[df['session_id'] == sid].sort_values('step')
    poses_all[i], acts_all[i], rtgs_all[i], masks_all[i] = pack_session_ids(sdf, FULL_CTX)

valid_counts = masks_all.sum(axis=1).astype(int)
print(f'sessions packed: {len(sids)}   valid-len mean={valid_counts.mean():.0f} '
      f'min={valid_counts.min()}  max={valid_counts.max()}')

pose_t = torch.from_numpy(poses_all)
act_t  = torch.from_numpy(acts_all)
rtg_t  = torch.from_numpy(rtgs_all)
mask_t = torch.from_numpy(masks_all)
ram_mb = sum(x.element_size() * x.numel() for x in (pose_t, act_t, rtg_t, mask_t)) / 1e6
print(f'dataset RAM ≈ {ram_mb:.0f} MB')

full = TensorDataset(rtg_t, pose_t, act_t, mask_t)
n = len(full); n_val = max(1, int(n * VAL_FRAC)); n_train = n - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full, [n_train, n_val], generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, pin_memory=(device == 'cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          drop_last=False, pin_memory=(device == 'cuda'))
print(f'sessions: train={n_train}  val={n_val}')


sessions packed: 422   valid-len mean=1784 min=259  max=3233
dataset RAM ≈ 19 MB
sessions: train=380  val=42


## 3. Train (loss masked on padding)


In [3]:
from tqdm.auto import tqdm

cfg = LinearDTConfig(
    embed_dim=EMBED_DIM, num_actions=5, context_size=FULL_CTX,
    num_heads=NUM_HEADS, num_layers=NUM_LAYERS, pos_encoding=POS_ENCODING,
)
model = LinearDecisionTransformer(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
model = torch.compile(model)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
ce = nn.CrossEntropyLoss(reduction='none')

USE_BF16 = (device == 'cuda')
print(f'params={n_params:,}  bf16={USE_BF16}  compiled=True')

def expand_batch(rtg_b, pose_b, act_b, mask_b):
    """(rtg, pose_id, action_id, mask) -> (rtg, state, action_oh, targets, mask) on device."""
    rtg_b  = rtg_b.unsqueeze(-1).to(device, non_blocking=True)            # (B, L, 1)
    pose_b = pose_b.to(device, non_blocking=True).long()                  # (B, L)
    act_b  = act_b.to(device, non_blocking=True).long()                   # (B, L)
    mask_b = mask_b.to(device, non_blocking=True)                         # (B, L)
    state  = pose_table[pose_b]                                           # (B, L, 60)
    action = nn.functional.one_hot(act_b, num_classes=5).float()          # (B, L, 5)
    return rtg_b, state, action, act_b, mask_b

def run_epoch(loader, train: bool, epoch_no: int):
    model.train() if train else model.eval()
    sum_loss = sum_correct = sum_tokens = 0.0
    desc = f'ep {epoch_no:3d}/{EPOCHS} ' + ('train' if train else 'val')
    pbar = tqdm(loader, desc=desc, leave=False)
    ctx_mgr = torch.enable_grad() if train else torch.no_grad()
    with ctx_mgr:
        for rtg, pose_ids, action_ids, mask in pbar:
            rtg, state, action, targets, mask = expand_batch(rtg, pose_ids, action_ids, mask)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16):
                logits = model(rtg, state, action)
                losses = ce(logits.reshape(-1, cfg.num_actions), targets.reshape(-1))
                losses = losses.reshape(targets.shape)
                denom = mask.sum().clamp_min(1.0)
                loss = (losses * mask).sum() / denom
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            sum_loss    += (losses * mask).sum().item()
            sum_correct += ((logits.argmax(-1) == targets).float() * mask).sum().item()
            sum_tokens  += mask.sum().item()
            if train and sum_tokens > 0:
                pbar.set_postfix(loss=f'{sum_loss/sum_tokens:.4f}',
                                 acc=f'{sum_correct/sum_tokens:.3f}')
    return sum_loss / max(sum_tokens, 1), sum_correct / max(sum_tokens, 1)

history = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    ep0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True,  epoch_no=epoch)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False, epoch_no=epoch)
    ep_sec = time.time() - ep0
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc, 'epoch_sec': ep_sec})
    print(f'[epoch {epoch:3d}] train_loss={train_loss:.4f} acc={train_acc:.3f} '
          f'| val_loss={val_loss:.4f} acc={val_acc:.3f}  '
          f'({ep_sec:.1f}s/epoch, {time.time()-t0:.0f}s total)')

ckpt = RUN_DIR / 'model.pt'
state_dict = (model._orig_mod if hasattr(model, '_orig_mod') else model).state_dict()
torch.save({'state_dict': state_dict, 'cfg': cfg.__dict__,
            'arch': 'linear_fullctx'}, ckpt)
(RUN_DIR / 'metrics.jsonl').write_text(''.join(json.dumps(h)+'\n' for h in history))
print(f'\nsaved checkpoint -> {ckpt}')


params=404,109  bf16=True  compiled=True


ep   1/5 train:   0%|          | 0/95 [00:00<?, ?it/s]

ep   1/5 val:   0%|          | 0/11 [00:00<?, ?it/s]

[epoch   1] train_loss=1.0910 acc=0.610 | val_loss=1.0117 acc=0.626  (33.2s/epoch, 33s total)


ep   2/5 train:   0%|          | 0/95 [00:00<?, ?it/s]

...

ep   4/5 val:   0%|          | 0/11 [00:00<?, ?it/s]

[epoch   4] train_loss=0.7695 acc=0.716 | val_loss=0.7729 acc=0.712  (3.9s/epoch, 45s total)


ep   5/5 train:   0%|          | 0/95 [00:00<?, ?it/s]

ep   5/5 val:   0%|          | 0/11 [00:00<?, ?it/s]

[epoch   5] train_loss=0.7292 acc=0.727 | val_loss=0.7368 acc=0.723  (3.9s/epoch, 49s total)

saved checkpoint -> ../runs/dt/nb08/model.pt


## 4. Inference movie

Grow a per-step (rtg, state, action) buffer over the rollout — the model conditions on the entire history each step. Truncated to the most recent `FULL_CTX` only if the rollout ever overflows.


In [5]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw

RTG_TARGET   = 20.0
MAX_STEPS    = 600
BASE_TEMP    = 1.0
ROLLOUT_SEED = 0
MOVIE_PATH   = RUN_DIR / 'rollout.mp4'
ACTION_NAMES = ['Left', 'Right', 'Forward', 'EnterWell', 'Pause']

model.eval()
env = CornerMazeEnv(session_type='exposure', obs_mode='view', render_mode='rgb_array')
env.reset(seed=ROLLOUT_SEED)

hist_state, hist_action, hist_rtg = [], [], []

frames = []
last_pose = None; stagnation = 0
total_reward = 0.0; n_well_rewards = 0

for step in range(MAX_STEPS):
    x, y, d = int(env.agent_pos[0]), int(env.agent_pos[1]), int(env.agent_dir)
    if (x, y, d) == last_pose: stagnation += 1
    else: stagnation = 0
    last_pose = (x, y, d)
    temp = BASE_TEMP + (stagnation // 3) * 1.5

    s_vec = encoder.encode(x, y, d).astype(np.float32)
    hist_state.append(s_vec)
    hist_rtg.append(np.array([RTG_TARGET], dtype=np.float32))
    if len(hist_action) < len(hist_state):
        hist_action.append(np.zeros(5, dtype=np.float32))   # placeholder for current step

    L = len(hist_state)
    if L > FULL_CTX:
        hist_state  = hist_state[-FULL_CTX:]
        hist_action = hist_action[-FULL_CTX:]
        hist_rtg    = hist_rtg[-FULL_CTX:]
        L = FULL_CTX
    rtg_t    = torch.from_numpy(np.stack(hist_rtg)).unsqueeze(0).to(device)
    state_t  = torch.from_numpy(np.stack(hist_state)).unsqueeze(0).to(device)
    action_t = torch.from_numpy(np.stack(hist_action)).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(rtg_t, state_t, action_t)
        probs = torch.softmax(logits[0, -1, :] / temp, dim=-1)
        action = int(torch.multinomial(probs, num_samples=1).item())

    img = env.render()
    canvas = Image.fromarray(img).resize((416, 416)).convert('RGB')
    panel = Image.new('RGB', (416, 464), 'black')
    panel.paste(canvas, (0, 48))
    draw = ImageDraw.Draw(panel)
    label = (f'step={step:3d} act={ACTION_NAMES[action]} '
             f'rtg={RTG_TARGET:.1f} ret={total_reward:+.2f} wells={n_well_rewards} L={L}')
    if stagnation > 0: label += f' stuck={stagnation} T={temp:.1f}'
    draw.text((6, 6),  label,                 fill='white')
    draw.text((6, 24), f'pose=({x},{y},{d})', fill='white')
    frames.append(np.array(panel))

    _, reward, term, trunc, _ = env.step(action)
    total_reward += reward
    if reward > 0.5: n_well_rewards += 1
    oh = np.zeros(5, dtype=np.float32); oh[action] = 1.0
    hist_action[-1] = oh
    if term or trunc:
        break

print(f'rollout: {len(frames)} steps, total_reward={total_reward:+.3f}, '
      f'wells={n_well_rewards}')
imageio.mimwrite(MOVIE_PATH, frames, fps=10, codec='libx264')
print(f'wrote -> {MOVIE_PATH}')


rollout: 600 steps, total_reward=+8.096, wells=8
wrote -> ../runs/dt/nb08/rollout.mp4


## 5. Watch the movie


In [6]:
from IPython.display import Video
Video(str(MOVIE_PATH), embed=False, width=500)
